In [1]:
%run data_extraction_feature_engineering.ipynb
df_train_full, df_test_full = extract_data()
VAL_START_DATE = '2012-09-01'

Loading data...
Train shape: (421570, 5)
Features shape: (8190, 12)
Stores shape: (45, 3)
Test shape: (115064, 4)
Merging datasets...

Data Preparation Complete!
X_train shape: (397841, 24)
X_val shape: (23729, 24)
   IsHoliday  Temperature  Fuel_Price  MarkDown1  MarkDown2  MarkDown3  \
0      False        42.31       2.572        0.0        0.0        0.0   
1       True        38.51       2.548        0.0        0.0        0.0   
2      False        39.93       2.514        0.0        0.0        0.0   
3      False        46.63       2.561        0.0        0.0        0.0   
4      False        46.50       2.625        0.0        0.0        0.0   

   MarkDown4  MarkDown5         CPI  Unemployment  ...  sales_lag_52  \
0        0.0        0.0  211.096358         8.106  ...           NaN   
1        0.0        0.0  211.242170         8.106  ...           NaN   
2        0.0        0.0  211.289143         8.106  ...           NaN   
3        0.0        0.0  211.319643         8.106  .

In [2]:
!pip install wandb -q
import wandb
import os
api_key = "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"
if api_key:
    wandb.login(key=api_key)
else:
    print("Warning: could not log in wandb ")


[notice] A new release of pip is available: 25.3 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip
wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: Appending key for api.wandb.ai to your netrc file: C:\Users\cotne\_netrc
wandb: Currently logged in as: slomi23 (slomi23-free-university-of-tbilisi-) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


## 1. Sliding-window sequences

In [3]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import mean_absolute_error
import matplotlib.pyplot as plt

torch.manual_seed(42)
np.random.seed(42)

LOOKBACK = 52
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", DEVICE)

Using device: cpu


In [4]:
def build_windows(df, lookback, val_start_date):
    df = df.sort_values(['Store', 'Dept', 'Date']).copy()
    df['IsHoliday'] = df['IsHoliday'].astype(int) if 'IsHoliday' in df.columns else 0

    train_X, train_y, train_holiday = [], [], []
    val_X, val_y, val_holiday, val_meta = [], [], [], []

    for (store, dept), g in df.groupby(['Store', 'Dept']):
        g = g.reset_index(drop=True)
        sales = g['Weekly_Sales'].values
        dates = g['Date'].values
        holidays = g['IsHoliday'].values

        if len(g) <= lookback:
            continue

        for t in range(lookback, len(g)):
            window = sales[t - lookback:t]
            target = sales[t]
            target_date = dates[t]
            is_holiday = holidays[t]

            if target_date < np.datetime64(val_start_date):
                train_X.append(window)
                train_y.append(target)
                train_holiday.append(is_holiday)
            else:
                val_X.append(window)
                val_y.append(target)
                val_holiday.append(is_holiday)
                val_meta.append((store, dept, target_date))

    return (np.array(train_X, dtype=np.float32), np.array(train_y, dtype=np.float32), np.array(train_holiday),
            np.array(val_X, dtype=np.float32), np.array(val_y, dtype=np.float32), np.array(val_holiday), val_meta)

train_X, train_y, train_holiday, val_X, val_y, val_holiday, val_meta = build_windows(
    df_train_full, LOOKBACK, VAL_START_DATE
)

print(f"Train windows: {train_X.shape}, Val windows: {val_X.shape}")

Train windows: (237717, 52), Val windows: (23366, 52)


## 2. ნორმალიზება


In [5]:
class SalesWindowDataset(Dataset):
    def __init__(self, X, y):
        self.X = X
        self.y = y
        self.scale = np.clip(np.abs(X).mean(axis=1, keepdims=True), 1.0, None)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        x = self.X[idx] / self.scale[idx]
        y = self.y[idx] / self.scale[idx][0]
        return torch.tensor(x, dtype=torch.float32), torch.tensor(y, dtype=torch.float32), torch.tensor(self.scale[idx][0], dtype=torch.float32)

train_ds = SalesWindowDataset(train_X, train_y)
val_ds = SalesWindowDataset(val_X, val_y)

BATCH_SIZE = 256
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False)

## 3. DLinear model

In [6]:
class MovingAvg(nn.Module):
    def __init__(self, kernel_size):
        super().__init__()
        self.kernel_size = kernel_size
        self.avg = nn.AvgPool1d(kernel_size=kernel_size, stride=1, padding=0)

    def forward(self, x):
        front = x[:, 0:1].repeat(1, (self.kernel_size - 1) // 2)
        end = x[:, -1:].repeat(1, (self.kernel_size - 1) // 2)
        x_padded = torch.cat([front, x, end], dim=1)
        return self.avg(x_padded.unsqueeze(1)).squeeze(1)


class DLinear(nn.Module):
    def __init__(self, lookback, horizon=1, kernel_size=25):
        super().__init__()
        self.moving_avg = MovingAvg(kernel_size)
        self.linear_trend = nn.Linear(lookback, horizon)
        self.linear_seasonal = nn.Linear(lookback, horizon)

    def forward(self, x):
        trend = self.moving_avg(x)
        seasonal = x - trend
        out = self.linear_trend(trend) + self.linear_seasonal(seasonal)
        return out.squeeze(-1)

model = DLinear(lookback=LOOKBACK, horizon=1, kernel_size=25).to(DEVICE)
print(model)

DLinear(
  (moving_avg): MovingAvg(
    (avg): AvgPool1d(kernel_size=(25,), stride=(1,), padding=(0,))
  )
  (linear_trend): Linear(in_features=52, out_features=1, bias=True)
  (linear_seasonal): Linear(in_features=52, out_features=1, bias=True)
)


## 4. ტრენინგი (ვალიდაცია WMAE-ით)

In [7]:
CONFIG = {
    "model_type": "DLinear",
    "run_name": "DLinear_baseline",
    "lookback": LOOKBACK,
    "kernel_size": 25,
    "batch_size": BATCH_SIZE,
    "lr": 1e-3,
    "max_epochs": 100,
    "early_stopping_patience": 10,
    "seed": 42,
}

run = wandb.init(
    project="walmart-sales-forecasting_Dlinear",
    group="DLinear_Training",
    name=CONFIG["run_name"],
    job_type="train",
    config=CONFIG,
)

optimizer = torch.optim.Adam(model.parameters(), lr=CONFIG["lr"])
loss_fn = nn.L1Loss()

def compute_wmae(y_true, y_pred, is_holiday):
    weights = np.where(is_holiday, 5, 1)
    return np.average(np.abs(y_true - y_pred), weights=weights)

def evaluate(loader, holiday_array):
    model.eval()
    preds, targets = [], []
    with torch.no_grad():
        for xb, yb, scale in loader:
            xb, yb, scale = xb.to(DEVICE), yb.to(DEVICE), scale.to(DEVICE)
            out = model(xb)
            preds.append((out * scale).cpu().numpy())
            targets.append((yb * scale).cpu().numpy())
    preds = np.concatenate(preds)
    targets = np.concatenate(targets)
    mae = mean_absolute_error(targets, preds)
    wmae = compute_wmae(targets, preds, holiday_array)
    return mae, wmae, preds, targets

best_val_wmae = np.inf
patience_counter = 0
best_state = None

for epoch in range(CONFIG["max_epochs"]):
    model.train()
    epoch_loss = 0.0
    for xb, yb, scale in train_loader:
        xb, yb = xb.to(DEVICE), yb.to(DEVICE)
        optimizer.zero_grad()
        out = model(xb)
        loss = loss_fn(out, yb)
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item() * len(xb)
    epoch_loss /= len(train_ds)

    train_mae, train_wmae, _, _ = evaluate(train_loader, train_holiday)
    val_mae, val_wmae, _, _ = evaluate(val_loader, val_holiday)

    wandb.log({
        "epoch": epoch,
        "train_loss": epoch_loss,
        "train_mae": train_mae,
        "train_wmae": train_wmae,
        "val_mae": val_mae,
        "val_wmae": val_wmae,
    })

    if epoch % 5 == 0 or epoch == CONFIG["max_epochs"] - 1:
        print(f"Epoch {epoch:3d} | loss={epoch_loss:.4f} | train_wmae={train_wmae:.2f} | val_wmae={val_wmae:.2f}")

    if val_wmae < best_val_wmae:
        best_val_wmae = val_wmae
        best_state = {k: v.clone() for k, v in model.state_dict().items()}
        patience_counter = 0
    else:
        patience_counter += 1
        if patience_counter >= CONFIG["early_stopping_patience"]:
            print(f"Early stopping at epoch {epoch} (best val WMAE={best_val_wmae:.2f})")
            break

model.load_state_dict(best_state)

Epoch   0 | loss=0.1825 | train_wmae=1645.29 | val_wmae=1426.59
Epoch   5 | loss=0.1701 | train_wmae=1641.95 | val_wmae=1424.59
Epoch  10 | loss=0.1700 | train_wmae=1676.74 | val_wmae=1473.82
Early stopping at epoch 14 (best val WMAE=1417.75)


<All keys matched successfully>

## 5. Final evaluation & logging

In [ ]:
train_mae, train_wmae, train_preds, train_targets = evaluate(train_loader, train_holiday)
val_mae, val_wmae, val_preds, val_targets = evaluate(val_loader, val_holiday)

if run:
    wandb.log({"final_validation_wmae": val_wmae, "final_validation_mae": val_mae})
    wandb.log({"train_wmae": train_wmae, "train_mae": train_mae})

    plt.figure(figsize=(10, 6))
    idx = np.argsort(val_targets)
    plt.plot(val_targets[idx], label="Actual", alpha=0.7)
    plt.plot(val_preds[idx], label="Predicted", alpha=0.7)
    plt.title("DLinear: Validation Predictions vs Actuals (sorted by actual value)")
    plt.legend()
    plt.tight_layout()
    wandb.log({"val_predictions_plot": wandb.Image(plt)})
    plt.close()

    trend_w = model.linear_trend.weight.detach().cpu().numpy().flatten()
    seasonal_w = model.linear_seasonal.weight.detach().cpu().numpy().flatten()
    plt.figure(figsize=(10, 4))
    plt.plot(trend_w, label="Trend weights")
    plt.plot(seasonal_w, label="Seasonal weights")
    plt.title("DLinear learned weights across lookback window")
    plt.xlabel("Weeks back")
    plt.legend()
    plt.tight_layout()
    wandb.log({"dlinear_weights_plot": wandb.Image(plt)})
    plt.close()

    artifact = wandb.Artifact("dlinear-model-final", type="model")
    torch.save(model.state_dict(), "dlinear_model.pt")
    artifact.add_file("dlinear_model.pt")
    run.log_artifact(artifact)

print(f"Final Train WMAE: {train_wmae:.2f}")
print(f"Final Train MAE: {train_mae:.2f}")
print(f"Final Validation WMAE: {val_wmae:.2f}")
print(f"Final Validation MAE: {val_mae:.2f}")

if run:
    wandb.finish()